# Goal
To test the CMI framework on real datasets. 

# Imports

In [2]:
import os
import numpy as np
import pandas as pd


In [3]:
def load_yeast_from_openml():
    """Load Yeast multi-label dataset via OpenML using sklearn.fetch_openml."""
    from sklearn.datasets import fetch_openml
    X, Y = fetch_openml("yeast", version=4, return_X_y=True, as_frame=False)
    # Y comes as string labels like "TRUE"/"FALSE" or as categorical; convert to binary indicator
    # In scikit-learn example, they do: Y = (Y == "TRUE")
    Y_bin = (Y == "TRUE").astype(int)
    return X, Y_bin

def load_cal500_cbar(data_home=None, codebook_size=512, download_if_missing=True):
    """Load CAL500 dataset via the CBAR library (if available)."""
    import cbar.datasets
    X, Y = cbar.datasets.fetch_cal500(
        data_home=data_home, download_if_missing=download_if_missing, codebook_size=codebook_size
    )
    # X is DataFrame or array, Y is DataFrame or array of shape (n_samples, n_labels)
    return np.array(X), np.array(Y)

def load_cal500_arff(mldr_path):
    """Alternate loader: read CAL500 ARFF + XML label metadata (Mulan format)."""
    import arff  # from liac-arff
    # Suppose we have cal500.arff and cal500.xml in mldr_path
    arff_fp = os.path.join(mldr_path, "cal500.arff")
    # Load ARFF
    with open(arff_fp, "r") as f:
        ar = arff.load(f)
    df = pd.DataFrame(ar["data"], columns=[attr[0] for attr in ar["attributes"]])
    # Assume that last 174 columns are labels; or parse xml to find label indices
    # Here we pick a naive split:
    n_labels = 174
    Y = df.iloc[:, -n_labels:].astype(int).to_numpy()
    X = df.iloc[:, :-n_labels].astype(float).to_numpy()
    return X, Y

In [4]:
X_yeast, Y_yeast = load_yeast_from_openml()
print("Yeast:", X_yeast.shape, Y_yeast.shape)
X_cal, Y_cal = load_cal500_cbar()
print("CAL500:", X_cal.shape, Y_cal.shape)

Yeast: (2417, 103) (2417, 14)


ModuleNotFoundError: No module named 'cbar'